In [0]:
dbutils.widgets.removeAll()

from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "retail_dev")
dbutils.widgets.text("esquema_origen", "silver")
dbutils.widgets.text("esquema_destino", "golden")

catalogo = dbutils.widgets.get("catalogo")
esquema_origen = dbutils.widgets.get("esquema_origen")
esquema_destino = dbutils.widgets.get("esquema_destino")

tabla_ventas = (
    f"{catalogo}.{esquema_origen}."
    "ventas_detalle_limpias"
)

tabla_inventario = (
    f"{catalogo}.{esquema_origen}."
    "inventario_limpio"
)

tabla_destino = (
    f"{catalogo}.{esquema_destino}."
    "resumen_retail"
)

print(f"Tabla de ventas: {tabla_ventas}")
print(f"Tabla de inventario: {tabla_inventario}")
print(f"Tabla Golden: {tabla_destino}")

In [0]:
df_ventas = spark.table(tabla_ventas)
df_inventario = spark.table(tabla_inventario)

print(
    f"Registros de ventas Silver: "
    f"{df_ventas.count()}"
)

print(
    f"Registros de inventario Silver: "
    f"{df_inventario.count()}"
)

In [0]:
print("Esquema de ventas:")
df_ventas.printSchema()

print("\nEsquema de inventario:")
df_inventario.printSchema()

In [0]:
df_resumen_ventas = (
    df_ventas

    .groupBy(
        "categoria",
        "estado_envio"
    )

    .agg(
        F.countDistinct(
            "orden_id"
        ).alias(
            "total_ordenes"
        ),

        F.countDistinct(
            "cliente_id"
        ).alias(
            "clientes_unicos"
        ),

        F.sum(
            "cantidad"
        ).alias(
            "unidades_vendidas"
        ),

        F.round(
            F.sum("importe_neto"),
            2
        ).alias(
            "venta_total"
        ),

        F.countDistinct(
            F.when(
                F.col("orden_retrasada") == "SI",
                F.col("orden_id")
            )
        ).alias(
            "ordenes_retrasadas"
        )
    )

    .withColumn(
        "ticket_promedio",
        F.when(
            F.col("total_ordenes") > 0,
            F.round(
                F.col("venta_total")
                / F.col("total_ordenes"),
                2
            )
        ).otherwise(
            F.lit(0.0)
        )
    )
)

print(
    f"Registros del resumen de ventas: "
    f"{df_resumen_ventas.count()}"
)

display(
    df_resumen_ventas
    .orderBy(
        "categoria",
        "estado_envio"
    )
)

In [0]:
df_resumen_inventario = (
    df_inventario

    .groupBy(
        "categoria"
    )

    .agg(
        F.countDistinct(
            F.when(
                F.col("estatus_inventario").isin(
                    "BAJO STOCK",
                    "SIN STOCK"
                ),
                F.col("producto_id")
            )
        ).alias(
            "productos_bajo_stock"
        )
    )
)

print(
    f"Categorías del resumen de inventario: "
    f"{df_resumen_inventario.count()}"
)

display(
    df_resumen_inventario
    .orderBy("categoria")
)

In [0]:
df_resumen_retail = (
    df_resumen_ventas.alias("v")

    .join(
        df_resumen_inventario.alias("i"),
        F.col("v.categoria")
        == F.col("i.categoria"),
        "left"
    )

    .select(
        F.col("v.categoria").alias("categoria"),

        F.col("v.estado_envio").alias(
            "estado_envio"
        ),

        F.col("v.total_ordenes")
        .cast("int")
        .alias("total_ordenes"),

        F.col("v.clientes_unicos")
        .cast("int")
        .alias("clientes_unicos"),

        F.col("v.unidades_vendidas")
        .cast("int")
        .alias("unidades_vendidas"),

        F.col("v.venta_total")
        .cast("double")
        .alias("venta_total"),

        F.col("v.ticket_promedio")
        .cast("double")
        .alias("ticket_promedio"),

        F.coalesce(
            F.col("i.productos_bajo_stock"),
            F.lit(0)
        )
        .cast("int")
        .alias("productos_bajo_stock"),

        F.col("v.ordenes_retrasadas")
        .cast("int")
        .alias("ordenes_retrasadas")
    )
)

print(
    f"Registros finales para Golden: "
    f"{df_resumen_retail.count()}"
)

display(
    df_resumen_retail
    .orderBy(
        "categoria",
        "estado_envio"
    )
)

In [0]:
df_resumen_retail.printSchema()

In [0]:
validacion_previa = (
    df_resumen_retail.select(
        F.count("*").alias(
            "registros"
        ),

        F.sum(
            F.when(
                F.col("categoria").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "categoria_nula"
        ),

        F.sum(
            F.when(
                F.col("estado_envio").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "estado_envio_nulo"
        ),

        F.sum(
            F.when(
                F.col("total_ordenes") <= 0,
                1
            ).otherwise(0)
        ).alias(
            "total_ordenes_invalido"
        ),

        F.sum(
            F.when(
                F.col("clientes_unicos") <= 0,
                1
            ).otherwise(0)
        ).alias(
            "clientes_unicos_invalido"
        ),

        F.sum(
            F.when(
                F.col("unidades_vendidas") <= 0,
                1
            ).otherwise(0)
        ).alias(
            "unidades_vendidas_invalido"
        ),

        F.sum(
            F.when(
                F.col("venta_total") < 0,
                1
            ).otherwise(0)
        ).alias(
            "venta_total_negativa"
        ),

        F.sum(
            F.when(
                F.col("ticket_promedio") < 0,
                1
            ).otherwise(0)
        ).alias(
            "ticket_promedio_negativo"
        ),

        F.sum(
            F.when(
                F.col("productos_bajo_stock") < 0,
                1
            ).otherwise(0)
        ).alias(
            "productos_bajo_stock_negativo"
        ),

        F.sum(
            F.when(
                F.col("ordenes_retrasadas") < 0,
                1
            ).otherwise(0)
        ).alias(
            "ordenes_retrasadas_negativo"
        )
    )
)

display(validacion_previa)

In [0]:
columnas_dataframe = df_resumen_retail.columns

columnas_tabla = [
    campo.name
    for campo in spark.table(tabla_destino).schema.fields
]

print("Columnas del DataFrame:")
print(columnas_dataframe)

print("\nColumnas de la tabla Golden:")
print(columnas_tabla)

if columnas_dataframe != columnas_tabla:
    raise ValueError(
        "Las columnas o su orden no coinciden con "
        "la tabla Golden."
    )

print(
    "Las columnas coinciden correctamente."
)

In [0]:
(
    df_resumen_retail.write
    .mode("overwrite")
    .insertInto(tabla_destino)
)

print(
    f"Tabla Golden cargada correctamente: "
    f"{tabla_destino}"
)

In [0]:
df_golden = spark.table(tabla_destino)

cantidad_dataframe = df_resumen_retail.count()
cantidad_golden = df_golden.count()

print(
    f"Registros del DataFrame Golden: "
    f"{cantidad_dataframe}"
)

print(
    f"Registros cargados en Golden: "
    f"{cantidad_golden}"
)

if cantidad_dataframe != cantidad_golden:
    raise ValueError(
        "La cantidad del DataFrame no coincide "
        "con la tabla Golden."
    )

print(
    "Carga de resumen_retail "
    "completada correctamente."
)

display(
    df_golden
    .orderBy(
        "categoria",
        "estado_envio"
    )
)

In [0]:
validacion_final = (
    df_golden.select(
        F.count("*").alias(
            "registros"
        ),

        F.sum(
            F.when(
                F.col("categoria").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "categoria_nula"
        ),

        F.sum(
            F.when(
                F.col("estado_envio").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "estado_envio_nulo"
        ),

        F.sum(
            F.when(
                F.col("total_ordenes") <= 0,
                1
            ).otherwise(0)
        ).alias(
            "ordenes_invalidas"
        ),

        F.sum(
            F.when(
                F.col("unidades_vendidas") <= 0,
                1
            ).otherwise(0)
        ).alias(
            "unidades_invalidas"
        ),

        F.sum(
            F.when(
                F.col("venta_total") < 0,
                1
            ).otherwise(0)
        ).alias(
            "venta_negativa"
        ),

        F.sum(
            F.when(
                F.col("ticket_promedio") < 0,
                1
            ).otherwise(0)
        ).alias(
            "ticket_negativo"
        ),

        F.sum(
            F.when(
                F.col("productos_bajo_stock") < 0,
                1
            ).otherwise(0)
        ).alias(
            "bajo_stock_negativo"
        ),

        F.sum(
            F.when(
                F.col("ordenes_retrasadas") < 0,
                1
            ).otherwise(0)
        ).alias(
            "retrasadas_negativas"
        )
    )
)

display(validacion_final)

In [0]:
resumen_general = (
    df_ventas.agg(
        F.countDistinct(
            "orden_id"
        ).alias(
            "total_ordenes"
        ),

        F.countDistinct(
            "cliente_id"
        ).alias(
            "clientes_unicos"
        ),

        F.sum(
            "cantidad"
        ).alias(
            "unidades_vendidas"
        ),

        F.round(
            F.sum("importe_neto"),
            2
        ).alias(
            "venta_total"
        ),

        F.countDistinct(
            F.when(
                F.col("orden_retrasada") == "SI",
                F.col("orden_id")
            )
        ).alias(
            "ordenes_retrasadas"
        )
    )
)

display(resumen_general)